# HeritageGuard AI: Policy Recommendation Agent (Colab)
Mount Carmel Church, Chimbel, Goa. Run the cells in order.

## 1. Install

In [1]:
!pip -q install -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 2. Upload the project zip
Upload `heritageguard-agent.zip` when asked.

In [4]:
from google.colab import files
files.upload()
!unzip -oq heritageguard-agent.zip
%cd heritageguard-agent

Saving .gitignore to .gitignore
Saving app.py to app.py
Saving HeritageGuard_Agent_Colab.ipynb to HeritageGuard_Agent_Colab.ipynb
Saving README.md to README.md
Saving requirements.txt to requirements.txt
/content/heritageguard-agent


In [5]:
!pwd

/content/heritageguard-agent


In [6]:
!ls

agent	data				 README.md
app.py	HeritageGuard_Agent_Colab.ipynb  requirements.txt


In [7]:
!ls agent data

agent:
heritageguard.py  __init__.py

data:
article_facts.json  policy_areas.json


## 3. Add your Gemini key
Get a key at aistudio.google.com. In Colab, click the key icon (Secrets) in the left bar, add a secret named `GEMINI_API_KEY`, paste the key, and switch on Notebook access. Never paste the key into a cell.

In [8]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
# Optional: os.environ['GEMINI_MODEL'] = 'a-model-name-from-AI-Studio'

## 4. Paste the article text
Replace the text between the quotes. The agent also uses `data/article_facts.json`.

In [9]:
ARTICLE = """
Govt boost for historic Mount Carmel site, locals for urgent conservation
Source: The Times of India (Goa)
Byline: Paul Fernandes

The article discusses the deteriorating condition of the historic Church of Our Lady of Mount Carmel in Chimbel, Goa, and the urgent need to conserve the remaining structure.

Built around 1749–1750 by the Tertiary Carmelites, the church is an example of Indo-Portuguese architecture in the Mannerist style. It once stood as an important religious and social landmark in the area. The article describes its historical significance and the threats facing the remains of the building.

The church has suffered extensive structural deterioration. Large portions of the structure and walls have collapsed, while vegetation growing around the ruins has contributed to the deterioration of the remaining facade. The site's neglected condition raises concerns about the loss of an important part of Goa's architectural and cultural heritage.

The government notified the site as a protected heritage monument in December 2020. However, protected status alone has not prevented further deterioration. Local residents and the Mount Carmel Conservation Association have called for urgent conservation measures to safeguard what remains of the church.

The article discusses proposals for structural preservation and the development of green space around the site. It also highlights the possibility of connecting Mount Carmel Church, located approximately 7 km from Old Goa, to the wider heritage tourism circuit.
"""

## 5. Run the agent

In [19]:
import os, time
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(timeout=30000),
)
names = [m.name.replace("models/", "") for m in client.models.list() if "flash" in m.name]
print(names)
for n in names:
    try:
        t = time.time()
        r = client.models.generate_content(model=n, contents="Say hello in five words.")
        print("WORKS:", n, f"({time.time() - t:.1f}s)")
    except Exception as e:
        print("failed:", n, str(e)[:80])

['gemini-2.5-flash', 'gemini-2.5-flash-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-3.1-flash-tts-preview', 'gemini-3.8-flash-tts', 'gemini-3.8-flash-lite-tts', 'gemini-2.5-flash-native-audio-latest', 'gemini-2.5-flash-native-audio-preview-09-2025', 'gemini-2.5-flash-native-audio-preview-12-2025', 'gemini-3.1-flash-live-preview']
failed: gemini-2.5-flash 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
failed: gemini-2.5-flash-preview-tts 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The requested combinat
failed: gemini-flash-latest 503

In [20]:
import os, importlib
os.environ['GEMINI_MODEL'] = 'gemini-3.5-flash'
import agent.heritageguard as h
importlib.reload(h)
result = h.run_agent(ARTICLE, progress=print)
print('Model used:', result['model'])

Step 1/3: analysing the issues
Step 2/3: generating policy options
Step 3/3: reviewing for unsupported claims
Model used: gemini-3.5-flash


## 6. View the recommendations (screenshot this for the exam)

In [21]:
from IPython.display import Markdown, display

def bl(title, items):
    return (f"**{title}**\n" + "\n".join(f"- {i}" for i in items) + "\n\n") if items else ""

a = result["analysis"]
out = "# Issue analysis\n"
out += "".join(f"- **{p['issue']}**: {p['evidence']}\n" for p in a["problems"])
out += "\n" + bl("Stakeholders", a["stakeholders"]) + bl("Priorities", a["priorities"])
out += "# Policy recommendations\n"
for p in result["policies"]:
    out += f"\n## {p['area']}: {p['recommendation']}\n{p['justification']}\n\n"
    out += bl("Evidence from the article", p["evidence_from_article"]) + bl("Assumptions", p["assumptions"])
    out += bl("Expected benefits", p["expected_benefits"]) + bl("Trade-offs", p["trade_offs"])
    out += bl("Implementation challenges", p["implementation_challenges"])
    out += bl("Success indicators", p["success_indicators"]) + bl("Evidence to verify", p["evidence_to_verify"])
out += "\n# Reviewer flags\n"
out += "".join(f"- **{f['policy']}**: {f['concern']}\n" for f in result["review_flags"]) or "No unsupported claims flagged (automated check; still review manually).\n"
display(Markdown(out))

# Issue analysis
- **Severe structural collapse and threat from natural elements**: Large portions of the church structure and adjoining walls have collapsed, debris lies around, and tree growth has destabilised the cracked facade.
- **Ongoing neglect despite official protected status**: The site was notified as a protected heritage monument in December 2020, but protected status alone has not halted deterioration or led to necessary structural interventions.

**Stakeholders**
- Government heritage authorities
- Mount Carmel Conservation Association (MCCAC)
- Local villagers and residents
- Civil engineers and structural assessors
- Heritage tourists

**Priorities**
- Commission a comprehensive qualified structural assessment prior to undertaking any restoration or preservation work.
- Implement urgent structural stabilization measures for the remaining facade and boundary wall.
- Remove destabilising vegetation and clear surrounding debris safely.
- Develop a conservation and green space rejuvenation plan to connect the site to the Old Goa heritage tourism circuit.

# Policy recommendations

## Conservation funding models: Establish a public-private partnership (PPP) model involving government heritage grants and corporate social responsibility (CSR) contributions.
The site requires immediate structural intervention beyond current government capacity, and a diversified funding stream ensures both initial stabilization and long-term maintenance.

**Evidence from the article**
- The government notified the site as a protected heritage monument in December 2020.
- The church needs immediate structural intervention.

**Assumptions**
- Government heritage grants are available for protected monuments.
- Corporate entities may be interested in funding heritage conservation as part of CSR initiatives.
- A PPP model can be legally structured for a state-protected monument.

**Expected benefits**
- Secures necessary capital for urgent structural stabilization.
- Reduces the financial burden on the state government.
- Ensures consistent funding for ongoing site maintenance.

**Trade-offs**
- Potential loss of full public control over site management if private partners demand branding rights.
- Complexity in coordinating between government agencies and private donors.

**Implementation challenges**
- Navigating bureaucratic requirements for public-private funding agreements.
- Attracting private donors to a site that currently lacks tourist infrastructure.

**Success indicators**
- Successful procurement of funds for the structural assessment.
- Completion of the initial stabilization phase.

**Evidence to verify**
- Availability of specific government heritage grants for the Chimbel site.


## Community-led stewardship: Formalize a Memorandum of Understanding (MoU) between the government and the Mount Carmel Conservation Association (MCCAC) to define roles in site monitoring and maintenance.
Local groups are already active in advocacy and have a vested interest in the site's preservation, making them ideal partners for daily monitoring and reporting.

**Evidence from the article**
- The MCCAC has pursued a long campaign for the site's heritage status.
- The MCCAC secretary says conserving the facade and boundary wall would give villagers a sit-out for prayers and Masses.

**Assumptions**
- The government is willing to delegate monitoring responsibilities to a local association.
- The MCCAC has the organizational capacity to manage regular site inspections.

**Expected benefits**
- Increased local ownership and protection of the site.
- Cost-effective monitoring and early detection of further deterioration.
- Alignment of conservation goals with community needs for prayer and social space.

**Trade-offs**
- Risk of conflict if community expectations for site use differ from professional conservation standards.
- Potential for volunteer burnout within the association.

**Implementation challenges**
- Defining the legal boundaries of the association's authority.
- Ensuring the association receives adequate training for heritage monitoring.

**Success indicators**
- Signing of a formal agreement between the government and the MCCAC.
- Regular submission of site condition reports by the association.

**Evidence to verify**
- Willingness of the MCCAC to take on formal maintenance responsibilities.


## Heritage-tourism integration: Develop a low-impact visitor management plan that limits foot traffic to designated pathways and provides interpretive signage without altering the ruins.
Connecting the site to the Old Goa circuit increases its visibility and potential for funding, but the fragile state of the ruins requires strict visitor control to prevent further damage.

**Evidence from the article**
- The site is barely 7 km from Old Goa and could be linked to the heritage circuit.
- The church has suffered extensive structural deterioration.

**Assumptions**
- The site can safely accommodate a limited number of visitors after structural stabilization.
- Tourism revenue can be ring-fenced for the site's conservation.

**Expected benefits**
- Increased public awareness of the site's historical significance.
- Potential for generating revenue to support ongoing maintenance.
- Integration into a broader, well-managed heritage tourism network.

**Trade-offs**
- Risk of increased wear and tear from tourist foot traffic.
- Potential for tourism to disrupt the tranquil environment valued by local residents.

**Implementation challenges**
- Designing infrastructure that is reversible and does not damage the historic fabric.
- Managing visitor numbers to prevent overcrowding.

**Success indicators**
- Successful inclusion of the site in official Old Goa heritage tourism maps.
- No increase in structural damage following the introduction of visitors.

**Evidence to verify**
- Feasibility of the site's structural integrity to support any level of tourism.


## Urban-planning buffers: Implement a designated 'heritage buffer zone' around the site where new construction is restricted in height and density.
The site is located in a fast-urbanizing landscape, and a buffer zone is necessary to protect the visual integrity and physical stability of the ruins from encroaching development.

**Evidence from the article**
- The ruins lie in a tranquil spot of a fast-urbanising landscape.

**Assumptions**
- Local planning authorities have the power to impose zoning restrictions around protected monuments.
- The current urban development plans can be amended to include heritage buffers.

**Expected benefits**
- Preservation of the site's tranquil setting.
- Prevention of structural damage caused by vibrations or excavation from nearby construction.
- Long-term protection of the site's visual landscape.

**Trade-offs**
- Potential opposition from landowners or developers restricted by the buffer zone.
- Economic impact on local property owners due to development limitations.

**Implementation challenges**
- Enforcing zoning regulations in a rapidly urbanizing area.
- Balancing heritage protection with the housing and development needs of the local community.

**Success indicators**
- Official notification of a heritage buffer zone in local planning documents.
- Zero unauthorized construction projects within the designated buffer area.

**Evidence to verify**
- Current status of urban planning regulations in the immediate vicinity of the church.


# Reviewer flags
- **Conservation funding models**: The justification claims that the required structural intervention is 'beyond current government capacity', which is a site fact not supported by the evidence.


## 7. Save the output

In [22]:
import json
with open('output.json', 'w') as f:
    json.dump(result, f, indent=2)
files.download('output.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gradio

In [24]:
!pip -q install gradio
from gradio_app import build_ui
build_ui().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6802e738b541719f02.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:
import agent.heritageguard as h
h.MODELS[:] = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.5-flash"]
print(h.MODELS)

['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3-flash-preview', 'gemini-3.5-flash']
